# IMPORT THƯ VIỆN

In [6]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# 1. Kết nối đến PostgreSQL để lấy dữ liệu master đã merge
DB_USER = "postgres"
DB_PASSWORD = "0101"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "coffee_db"

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# Tạo Lag Features

In [3]:
# Truy vấn toàn bộ dữ liệu từ bảng master trên PostgreSQL
query = "SELECT * FROM coffee_master_data ORDER BY \"Date\" ASC;"
df_lag = pd.read_sql(query, engine)

# Đảm bảo cột Date đúng định dạng datetime và sắp xếp tăng dần theo thời gian
df_lag['Date'] = pd.to_datetime(df_lag['Date'])
df_lag = df_lag.sort_values('Date').reset_index(drop=True)

print(f"[+] Đã tải thành công {len(df_lag)} dòng dữ liệu từ PostgreSQL bảng 'coffee_master_data'.")

# 2. Định nghĩa các cột cần tạo biến trễ (Lag)
columns_to_lag = [
    "Price_DakLak", "Price_DakNong", "Price_GiaLai", "Price_LamDong",
    "Robusta_Close", "Robusta_Volume", "USD_VND", 
    "Temp_Average", "Precipitation"
]

# Các mốc thời gian trễ (ngày) muốn tạo
lag_days = [1, 2, 3, 5, 7, 14, 30]

# 3. Vòng lặp tự động tạo các cột Lag Features
for col in columns_to_lag:
    if col in df_lag.columns:
        for lag in lag_days:
            new_col_name = f"{col}_Lag_{lag}"
            df_lag[new_col_name] = df_lag[col].shift(lag)

# 4. Loại bỏ các dòng chứa giá trị NaN do quá trình shift
df_lag_cleaned = df_lag.dropna().reset_index(drop=True)

# 5. Lưu kết quả ngược lại vào PostgreSQL thành một bảng mới (ví dụ: 'coffee_features_data') 
# hoặc lưu ra file CSV tùy chọn phục vụ cho bước train model
feature_table_name = "coffee_features_data"
df_lag_cleaned.to_sql(feature_table_name, engine, if_exists='replace', index=False)

print("-" * 69)
print("              TẠO LAG FEATURES TỪ POSTGRESQL THÀNH CÔNG")
print("-" * 69)
print(f" - Tổng số dòng hợp lệ sau khi tạo lag: {len(df_lag_cleaned)} ngày.")
print(f" - Tổng số cột đặc trưng:                {len(df_lag_cleaned.columns)} cột.")
print(f" - Đã đẩy bảng mới vào PostgreSQL:        '{feature_table_name}'")
print("-" * 69)

# Kiểm tra trực quan kết quả các cột Lag vừa tạo
sample_cols = ['Date', 'Price_DakLak', 'Price_DakLak_Lag_1', 'Price_DakLak_Lag_7']
display(df_lag_cleaned[sample_cols].head())

[+] Đã tải thành công 2090 dòng dữ liệu từ PostgreSQL bảng 'coffee_master_data'.
---------------------------------------------------------------------
              TẠO LAG FEATURES TỪ POSTGRESQL THÀNH CÔNG
---------------------------------------------------------------------
 - Tổng số dòng hợp lệ sau khi tạo lag: 2057 ngày.
 - Tổng số cột đặc trưng:                81 cột.
 - Đã đẩy bảng mới vào PostgreSQL:        'coffee_features_data'
---------------------------------------------------------------------


,Date,Price_DakLak,Price_DakLak_Lag_1,Price_DakLak_Lag_7
0,2021-02-03,31300.0,31800.0,31400.0
1,2021-02-04,31800.0,31300.0,31500.0
2,2021-02-05,32100.0,31800.0,31100.0
3,2021-02-06,31900.0,32100.0,31400.0
4,2021-02-07,31800.0,31900.0,31500.0


# Tạo Rolling Statistics

In [4]:
# Truy vấn dữ liệu từ bảng master trên PostgreSQL
query = "SELECT * FROM coffee_master_data ORDER BY \"Date\" ASC;"
df_rolling = pd.read_sql(query, engine)

# Đảm bảo cột Date đúng định dạng datetime và sắp xếp tăng dần
df_rolling['Date'] = pd.to_datetime(df_rolling['Date'])
df_rolling = df_rolling.sort_values('Date').reset_index(drop=True)

print(f"[+] Đã tải thành công {len(df_rolling)} dòng dữ liệu từ PostgreSQL.")

# 2. Định nghĩa các cột cần tính Rolling Statistics
columns_to_roll = [
    "Price_DakLak", "Price_DakNong", "Price_GiaLai", "Price_LamDong",
    "Robusta_Close", "Robusta_Volume", "USD_VND", "Temp_Average"
]

# Các khoảng cửa sổ thời gian trượt (window size) theo ngày
# Ví dụ: 7 ngày (1 tuần), 14 ngày (2 tuần), 30 ngày (1 tháng)
windows = [7, 14, 30]

# 3. Tính toán Rolling Statistics
for col in columns_to_roll:
    if col in df_rolling.columns:
        for w in windows:
            # LUÔN PHẢI DÙNG .shift(1) TRƯỚC KHI ROLLING ĐỂ TRÁNH DATA LEAKAGE!
            # Ý nghĩa: Chỉ tính trung bình/độ biến động của các ngày TRƯỚC ngày hiện tại.
            shifted_col = df_rolling[col].shift(1)
            
            # Trung bình trượt (Moving Average - xu hướng giá)
            df_rolling[f"{col}_RollMean_{w}"] = shifted_col.rolling(window=w).mean()
            
            # Độ lệch chuẩn trượt (Volatility - mức độ biến động/rủi ro)
            df_rolling[f"{col}_RollStd_{w}"] = shifted_col.rolling(window=w).std()
            
            # Giá cao nhất / thấp nhất trong cửa sổ trượt
            df_rolling[f"{col}_RollMax_{w}"] = shifted_col.rolling(window=w).max()
            df_rolling[f"{col}_RollMin_{w}"] = shifted_col.rolling(window=w).min()

# 4. Loại bỏ các dòng chứa giá trị NaN do quá trình rolling (các ngày đầu chuỗi chưa đủ dữ liệu cửa sổ)
df_rolling_cleaned = df_rolling.dropna().reset_index(drop=True)

# 5. Lưu kết quả ngược lại vào PostgreSQL thành bảng mới
rolling_table_name = "coffee_rolling_features_data"
df_rolling_cleaned.to_sql(rolling_table_name, engine, if_exists='replace', index=False)

print("-" * 69)
print("              TẠO ROLLING STATISTICS TỪ POSTGRESQL THÀNH CÔNG")
print("-" * 69)
print(f" - Tổng số dòng hợp lệ sau khi xử lý: {len(df_rolling_cleaned)} ngày.")
print(f" - Tổng số cột đặc trưng sau mở rộng: {len(df_rolling_cleaned.columns)} cột.")
print(f" - Đã đẩy bảng mới vào PostgreSQL:     '{rolling_table_name}'")
print("-" * 69)

# Kiểm tra trực quan kết quả các cột Rolling vừa tạo
sample_cols = ['Date', 'Price_DakLak', 'Price_DakLak_RollMean_7', 'Price_DakLak_RollStd_7']
display(df_rolling_cleaned[sample_cols].head())

[+] Đã tải thành công 2090 dòng dữ liệu từ PostgreSQL.
---------------------------------------------------------------------
              TẠO ROLLING STATISTICS TỪ POSTGRESQL THÀNH CÔNG
---------------------------------------------------------------------
 - Tổng số dòng hợp lệ sau khi xử lý: 2057 ngày.
 - Tổng số cột đặc trưng sau mở rộng: 114 cột.
 - Đã đẩy bảng mới vào PostgreSQL:     'coffee_rolling_features_data'
---------------------------------------------------------------------


,Date,Price_DakLak,Price_DakLak_RollMean_7,Price_DakLak_RollStd_7
0,2021-02-03,31300.0,31457.142857,207.019668
1,2021-02-04,31800.0,31442.857143,214.919697
2,2021-02-05,32100.0,31485.714286,254.483604
3,2021-02-06,31900.0,31628.571429,281.154084
4,2021-02-07,31800.0,31700.000000,276.887462


# Tạo Temporal Features

In [7]:
# Truy vấn dữ liệu từ bảng master trên PostgreSQL
query = "SELECT * FROM coffee_master_data ORDER BY \"Date\" ASC;"
df_temporal = pd.read_sql(query, engine)

# Đảm bảo cột Date đúng định dạng datetime và sắp xếp tăng dần
df_temporal['Date'] = pd.to_datetime(df_temporal['Date'])
df_temporal = df_temporal.sort_values('Date').reset_index(drop=True)

print(f"[+] Đã tải thành công {len(df_temporal)} dòng dữ liệu từ PostgreSQL.")

# 2. Tạo các đặc trưng thời gian cơ bản (Temporal Features)
df_temporal['Year'] = df_temporal['Date'].dt.year
df_temporal['Month'] = df_temporal['Date'].dt.month
df_temporal['Day'] = df_temporal['Date'].dt.day
df_temporal['DayOfWeek'] = df_temporal['Date'].dt.dayofweek  # Thứ từ 0 (Thứ 2) đến 6 (Chủ nhật)
df_temporal['DayOfYear'] = df_temporal['Date'].dt.dayofyear # Ngày thứ mấy trong năm (1-365)
df_temporal['Quarter'] = df_temporal['Date'].dt.quarter    # Quý trong năm (1-4)
df_temporal['Is_Weekend'] = df_temporal['DayOfWeek'].isin([5, 6]).astype(int) # Cờ đánh dấu ngày cuối tuần

# 3. Mã hóa lượng giác (Cyclical Encoding) cho chu kỳ thời gian
# Giúp thuật toán cây (LightGBM/XGBoost) hiểu rằng Tháng 12 và Tháng 1 nằm liền kề nhau về mặt thời gian (tính tuần hoàn)
df_temporal['Month_Sin'] = np.sin(2 * np.pi * df_temporal['Month'] / 12)
df_temporal['Month_Cos'] = np.cos(2 * np.pi * df_temporal['Month'] / 12)

df_temporal['DayOfYear_Sin'] = np.sin(2 * np.pi * df_temporal['DayOfYear'] / 365)
df_temporal['DayOfYear_Cos'] = np.cos(2 * np.pi * df_temporal['DayOfYear'] / 365)

# 4. Lưu kết quả ngược lại vào PostgreSQL thành bảng mới
temporal_table_name = "coffee_temporal_features_data"
df_temporal.to_sql(temporal_table_name, engine, if_exists='replace', index=False)

print("-" * 69)
print("              TẠO TEMPORAL FEATURES TỪ POSTGRESQL THÀNH CÔNG")
print("-" * 69)
print(f" - Tổng số dòng xử lý:                  {len(df_temporal)} ngày.")
print(f" - Tổng số cột đặc trưng sau mở rộng:   {len(df_temporal.columns)} cột.")
print(f" - Đã đẩy bảng mới vào PostgreSQL:       '{temporal_table_name}'")
print("-" * 69)

# Kiểm tra trực quan kết quả các cột Temporal vừa tạo
sample_cols = ['Date', 'Month', 'Quarter', 'Month_Sin', 'Month_Cos', 'Is_Weekend']
display(df_temporal[sample_cols].head())

[+] Đã tải thành công 2090 dòng dữ liệu từ PostgreSQL.
---------------------------------------------------------------------
              TẠO TEMPORAL FEATURES TỪ POSTGRESQL THÀNH CÔNG
---------------------------------------------------------------------
 - Tổng số dòng xử lý:                  2090 ngày.
 - Tổng số cột đặc trưng sau mở rộng:   29 cột.
 - Đã đẩy bảng mới vào PostgreSQL:       'coffee_temporal_features_data'
---------------------------------------------------------------------


,Date,Month,Quarter,Month_Sin,Month_Cos,Is_Weekend
0,2021-01-01,1,1,0.5,0.866025,0
1,2021-01-02,1,1,0.5,0.866025,1
2,2021-01-03,1,1,0.5,0.866025,1
3,2021-01-04,1,1,0.5,0.866025,0
4,2021-01-05,1,1,0.5,0.866025,0


# Tạo Target Variable

In [8]:
# Truy vấn dữ liệu từ bảng master trên PostgreSQL
query = "SELECT * FROM coffee_master_data ORDER BY \"Date\" ASC;"
df_target = pd.read_sql(query, engine)

# Đảm bảo cột Date đúng định dạng datetime và sắp xếp tăng dần
df_target['Date'] = pd.to_datetime(df_target['Date'])
df_target = df_target.sort_values('Date').reset_index(drop=True)

print(f"[+] Đã tải thành công {len(df_target)} dòng dữ liệu từ PostgreSQL.")

# 2. Định nghĩa cột mục tiêu chính cần dự đoán
target_col = "Price_DakLak"

# 3. Tạo Target cho các mốc thời gian trong tương lai (Forecast Horizons)
# - Dự đoán giá 7 ngày tới (Short-term)
# - Dự đoán giá 30 ngày tới (Medium-term)
horizons = [7, 30]

for h in horizons:
    # Dùng hàm shift âm (-h) để lấy giá trị của h ngày sau kéo về dòng hiện tại
    df_target[f"Target_Price_{h}d_Ahead"] = df_target[target_col].shift(-h)

# 4. Loại bỏ các dòng bị NaN ở cuối chuỗi do hàm shift(-h) (vì các ngày cuối không có dữ liệu tương lai để làm nhãn)
df_ml_ready = df_target.dropna(subset=[f"Target_Price_{h}d_Ahead" for h in horizons]).reset_index(drop=True)

# 5. Lưu bảng dữ liệu hoàn chỉnh cuối cùng (đã có Target) vào PostgreSQL
final_ml_table_name = "coffee_master_ml_dataset"
df_ml_ready.to_sql(final_ml_table_name, engine, if_exists='replace', index=False)

# Đồng thời lưu ra một bản file CSV cục bộ để sẵn sàng train mô hình ML
final_csv_path = "coffee/coffee_master_ml_dataset.csv"
df_ml_ready.to_csv(final_csv_path, index=False)

print("-" * 69)
print("                  TẠO TARGET VÀ HOÀN THIỆN DATASET THÀNH CÔNG")
print("-" * 69)
print(f" - Tổng số mẫu dữ liệu sẵn sàng train ML: {len(df_ml_ready)} dòng.")
print(f" - Tổng số cột đặc trưng + mục tiêu:      {len(df_ml_ready.columns)} cột.")
print(f" - Đã lưu bảng chuẩn vào PostgreSQL:       '{final_ml_table_name}'")
print(f" - Đã lưu file backup cục bộ:              '{final_csv_path}'")
print("-" * 69)

# Kiểm tra trực quan các cột Target vừa tạo ở phần cuối dataset (nơi có đủ dữ liệu tương lai)
sample_cols = ['Date', 'Price_DakLak', 'Target_Price_7d_Ahead', 'Target_Price_30d_Ahead']
display(df_ml_ready[sample_cols].tail())

[+] Đã tải thành công 2090 dòng dữ liệu từ PostgreSQL.
---------------------------------------------------------------------
                  TẠO TARGET VÀ HOÀN THIỆN DATASET THÀNH CÔNG
---------------------------------------------------------------------
 - Tổng số mẫu dữ liệu sẵn sàng train ML: 2060 dòng.
 - Tổng số cột đặc trưng + mục tiêu:      20 cột.
 - Đã lưu bảng chuẩn vào PostgreSQL:       'coffee_master_ml_dataset'
 - Đã lưu file backup cục bộ:              'coffee/coffee_master_ml_dataset.csv'
---------------------------------------------------------------------


,Date,Price_DakLak,Target_Price_7d_Ahead,Target_Price_30d_Ahead
2055,2026-08-18,95000.0,96500.0,94500.0
2056,2026-08-19,98800.0,96500.0,94200.0
2057,2026-08-20,98300.0,96500.0,93600.0
2058,2026-08-21,97700.0,96200.0,93600.0
2059,2026-08-22,97900.0,95200.0,93600.0


# Interaction Features & Differencing/Returns

In [9]:
# Truy vấn dữ liệu từ bảng master trên PostgreSQL
query = "SELECT * FROM coffee_master_data ORDER BY \"Date\" ASC;"
df_adv = pd.read_sql(query, engine)

# Đảm bảo cột Date đúng định dạng datetime và sắp xếp tăng dần
df_adv['Date'] = pd.to_datetime(df_adv['Date'])
df_adv = df_adv.sort_values('Date').reset_index(drop=True)

print(f"[+] Đã tải thành công {len(df_adv)} dòng dữ liệu từ PostgreSQL.")

# 2. Tạo Interaction Features (Đặc trưng tương tác vĩ mô)
# Quy đổi giá Robusta quốc tế sang VNĐ để thấy sự liên thông thị trường
# (Lưu ý: Tuỳ thuộc vào đơn vị quy đổi của sàn, ở đây ta tạo biến tích hợp tỷ giá x giá quốc tế)
df_adv['Robusta_VND_Equivalent'] = df_adv['Robusta_Close'] * df_adv['USD_VND']

# Chênh lệch giá giữa Đắk Lắk và Lâm Đồng (Đại diện cho spread vùng miền)
if 'Price_DakLak' in df_adv.columns and 'Price_LamDong' in df_adv.columns:
    df_adv['Spread_DakLak_LamDong'] = df_adv['Price_DakLak'] - df_adv['Price_LamDong']

# 3. Tạo Differencing & Returns Features (Sai phân và tỷ lệ thay đổi giá)
# Giúp mô hình bắt được xu hướng tăng/giảm mạnh (momentum)
price_cols_to_diff = ["Price_DakLak", "Price_DakNong", "Price_GiaLai", "Price_LamDong", "Robusta_Close"]
for col in price_cols_to_diff:
    if col in df_adv.columns:
        # Sai phân tuyệt đối so với ngày hôm trước (t - t-1)
        df_adv[f"{col}_Diff_1d"] = df_adv[col].diff(1)
        # Tỷ lệ phần trăm thay đổi so với ngày hôm trước (Return)
        df_adv[f"{col}_PctChange_1d"] = df_adv[col].pct_change(1)

# 4. Tạo lại Target dự đoán 7 ngày và 30 ngày tới cho Đắk Lắk
horizons = [7, 30]
for h in horizons:
    df_adv[f"Target_Price_{h}d_Ahead"] = df_adv["Price_DakLak"].shift(-h)

# 5. Làm sạch giá trị NaN phát sinh từ quá trình diff và shift
df_adv_cleaned = df_adv.dropna().reset_index(drop=True)

# 6. Lưu bảng đặc trưng nâng cao hoàn chỉnh vào PostgreSQL
advanced_table_name = "coffee_advanced_features_dataset"
df_adv_cleaned.to_sql(advanced_table_name, engine, if_exists='replace', index=False)

# Lưu bản backup CSV cục bộ
advanced_csv_path = "coffee/coffee_advanced_features_dataset.csv"
df_adv_cleaned.to_csv(advanced_csv_path, index=False)

print("-" * 69)
print("             TẠO ADVANCED FEATURES NÂNG CAO THÀNH CÔNG")
print("-" * 69)
print(f" - Tổng số mẫu dữ liệu sẵn sàng: {len(df_adv_cleaned)} dòng.")
print(f" - Tổng số cột đặc trưng mở rộng: {len(df_adv_cleaned.columns)} cột.")
print(f" - Đã lưu bảng vào PostgreSQL:   '{advanced_table_name}'")
print(f" - Đã lưu file backup cục bộ:    '{advanced_csv_path}'")
print("-" * 69)

# Kiểm tra trực quan các cột vừa tạo
sample_cols = ['Date', 'Price_DakLak', 'Robusta_VND_Equivalent', 'Price_DakLak_PctChange_1d', 'Target_Price_7d_Ahead']
display(df_adv_cleaned[sample_cols].tail())

[+] Đã tải thành công 2090 dòng dữ liệu từ PostgreSQL.
---------------------------------------------------------------------
             TẠO ADVANCED FEATURES NÂNG CAO THÀNH CÔNG
---------------------------------------------------------------------
 - Tổng số mẫu dữ liệu sẵn sàng: 2056 dòng.
 - Tổng số cột đặc trưng mở rộng: 32 cột.
 - Đã lưu bảng vào PostgreSQL:   'coffee_advanced_features_dataset'
 - Đã lưu file backup cục bộ:    'coffee/coffee_advanced_features_dataset.csv'
---------------------------------------------------------------------


,Date,Price_DakLak,Robusta_VND_Equivalent,Price_DakLak_PctChange_1d,Target_Price_7d_Ahead
2051,2026-08-18,95000.0,9.521080e+06,0.003168,96500.0
2052,2026-08-19,98800.0,9.409912e+06,0.040000,96500.0
2053,2026-08-20,98300.0,9.518685e+06,-0.005061,96500.0
2054,2026-08-21,97700.0,9.363016e+06,-0.006104,96200.0
2055,2026-08-22,97900.0,9.363016e+06,0.002047,95200.0


In [ ]:
print("[*] Đang tải các thành phần đặc trưng từ PostgreSQL để hợp nhất...")

# 2. Truy vấn lại từng bảng đặc trưng đã tạo trước đó qua cột 'Date'
df_master = pd.read_sql("SELECT * FROM coffee_master_data", engine)
df_lag = pd.read_sql("SELECT * FROM coffee_features_data", engine)
df_rolling = pd.read_sql("SELECT * FROM coffee_rolling_features_data", engine)
df_temporal = pd.read_sql("SELECT * FROM coffee_temporal_features_data", engine)
df_adv = pd.read_sql("SELECT * FROM coffee_advanced_features_dataset", engine)

# Đảm bảo cột Date chuẩn datetime trên tất cả các dataframe
for df in [df_master, df_lag, df_rolling, df_temporal, df_adv]:
    df['Date'] = pd.to_datetime(df['Date'])

# 3. Tiến hành Merge toàn bộ qua cột khóa 'Date'
# Bắt đầu từ bảng master, lần lượt join các bảng đặc trưng lại
df_full_features = df_master.merge(df_lag.drop(columns=[c for c in df_lag.columns if c in df_master.columns and c != 'Date']), on='Date', how='inner')
df_full_features = df_full_features.merge(df_rolling.drop(columns=[c for c in df_rolling.columns if c in df_full_features.columns and c != 'Date']), on='Date', how='inner')
df_full_features = df_full_features.merge(df_temporal.drop(columns=[c for c in df_temporal.columns if c in df_full_features.columns and c != 'Date']), on='Date', how='inner')
df_full_features = df_full_features.merge(df_adv.drop(columns=[c for c in df_adv.columns if c in df_full_features.columns and c != 'Date']), on='Date', how='inner')

# Sắp xếp theo thời gian tăng dần
df_full_features = df_full_features.sort_values('Date').reset_index(drop=True)

# 4. Lưu bảng Feature Store hoàn chỉnh cuối cùng vào PostgreSQL
master_store_name = "coffee_master_feature_store"
df_full_features.to_sql(master_store_name, engine, if_exists='replace', index=False)

print("-" * 69)
print("             HỢP NHẤT MASTER FEATURE STORE THÀNH CÔNG")
print("-" * 69)
print(f" - Tổng số dòng dữ liệu: {len(df_full_features)} ngày.")
print(f" - Tổng số cột đặc trưng: {len(df_full_features.columns)} cột (Bao gồm Lag, Rolling, Temporal, Advanced).")
print(f" - Đã lưu bảng chuẩn vào PostgreSQL: '{master_store_name}'")
print("-" * 69)

# Kiểm tra danh sách các nhóm cột hoặc hiển thị xem thử
print("\n[+] Các cột có sẵn trong Master Feature Store:")
print(df_full_features.columns.tolist()[:15], "...") # In 15 cột đầu làm mẫu

[*] Đang tải các thành phần đặc trưng từ PostgreSQL để hợp nhất...
---------------------------------------------------------------------
             HỢP NHẤT MASTER FEATURE STORE THÀNH CÔNG
---------------------------------------------------------------------
 - Tổng số dòng dữ liệu: 2027 ngày.
 - Tổng số cột đặc trưng: 202 cột (Bao gồm Lag, Rolling, Temporal, Advanced).
 - Đã lưu bảng chuẩn vào PostgreSQL: 'coffee_master_feature_store'
---------------------------------------------------------------------

[+] Các cột có sẵn trong Master Feature Store:
['Date', 'Price_DakLak', 'Price_DakNong', 'Price_GiaLai', 'Price_LamDong', 'Temp_Average', 'Temp_Min', 'Temp_Max', 'Precipitation', 'Wind_Speed', 'Wind_Direction', 'Solar_Radiation', 'USD_VND', 'Robusta_Open', 'Robusta_High'] ...
